## O que é o Indicador Criança Alfabetizada

Links / referências<br/>
https://www.gov.br/mec/pt-br (política educacional)<br/>
Dados: camada **Gold** do datalake da Fase 2 — `gold/br_inep_alfabetizacao/`

O Indicador Criança Alfabetizada acompanha se crianças em idade escolar atingem o nível esperado de alfabetização. Na Fase 3 usamos a Gold já integrada (aluno, território, metas e contexto socioeconômico) para **prever** se um aluno será considerado alfabetizado (`alfabetizado` = 0 ou 1) e gerar inteligência para gestores públicos.

Trabalhamos com quatro tabelas Gold:
- **`alunos_features`** — fato por aluno (target da modelagem)
- **`contexto_territorio`** — município × rede (PIB, IVS, lags)
- **`indicador_crianca_alfabetizada_municipio`** — taxas e metas municipais
- **`indicador_crianca_alfabetizada_uf`** — taxas e metas por UF

**Pergunta Norteadora:**  
*"Quais fatores educacionais, territoriais e socioeconômicos estão associados à alfabetização do aluno, e como isso apoia a escolha de features e o cuidado com data leakage na modelagem?"

In [1]:
# Libs
from pathlib import Path
import importlib
import sys
import warnings

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Recarrega módulos após alterações no código (evita cache do kernel)
for mod in ("src.config", "src.visualization.eda", "src.preprocessing.load_s3"):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

import src.config as config
importlib.reload(config)

from src.config import DATALAKE_BUCKET, EDA_N_ROWS, GOLD_PREFIX, GOLD_TABLES, GOLD_YEAR, IMAGES_DIR, LEAKAGE_COLS, TARGET_COL
from src.visualization.eda import (
    build_eda_frame,
    correlation_with_target,
    feature_columns,
    load_eda_data,
    profile,
    split_feature_types,
    target_balance,
    write_eda_report,
)

pd.set_option("display.max_columns", 40)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Fonte: s3://{DATALAKE_BUCKET}/{GOLD_PREFIX} | ano={GOLD_YEAR} | n_alunos={EDA_N_ROWS}")


Fonte: s3://tech-challenge-2-datalake-prod/gold/br_inep_alfabetizacao | ano=2024 | n_alunos=5000


### Leitura da base (Gold no S3)

Leitura **sempre** do datalake AWS:

`s3://{DATALAKE_BUCKET}/gold/br_inep_alfabetizacao/{tabela}/ano={GOLD_YEAR}/`

- **`alunos_features`:** primeiras `EDA_N_ROWS` linhas (peek, sem baixar tudo)
- **Demais tabelas:** partição completa do ano (são pequenas)

Credenciais AWS no `.env` (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, região).

In [2]:
tables = load_eda_data()
alunos = tables["alunos_features"]
contexto = tables["contexto_territorio"]
ind_mun = tables["indicador_crianca_alfabetizada_municipio"]
ind_uf = tables["indicador_crianca_alfabetizada_uf"]

alunos.head()

,id_aluno,id_municipio,rede,serie,alfabetizado,peso_aluno,_ingestion_timestamp,_silver_processed_at,_silver_batch_id,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,_source_table,_batch_id,nome_municipio,...,lag1_proporcao_aluno_nivel_2,lag1_proporcao_aluno_nivel_3,lag1_proporcao_aluno_nivel_4,lag1_proporcao_aluno_nivel_5,lag1_proporcao_aluno_nivel_6,lag1_proporcao_aluno_nivel_7,lag1_proporcao_aluno_nivel_8,lag1_uf_taxa_alfabetizacao,lag1_uf_media_portugues,lag1_uf_proporcao_aluno_nivel_0,lag1_uf_proporcao_aluno_nivel_1,lag1_uf_proporcao_aluno_nivel_2,lag1_uf_proporcao_aluno_nivel_3,lag1_uf_proporcao_aluno_nivel_4,lag1_uf_proporcao_aluno_nivel_5,lag1_uf_proporcao_aluno_nivel_6,lag1_uf_proporcao_aluno_nivel_7,lag1_uf_proporcao_aluno_nivel_8,_gold_processed_at,_gold_batch_id
0,21003592,2111201,municipal,2,1.0,1.310000,2026-08-21T14:53:54.662279+00:00,2026-09-01T22:43:25.497453+00:00,09680cbf-1ca4-4aa0-aad4-93f80d0fc395,65.19,68.00,70.69,73.24,75.65,77.90,80.0,2.0,basedosdados.br_inep_avaliacao_alfabetizacao.m...,639d5481-b968-4c35-b649-4a48b1d0da10,São José de Ribamar,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.39,749.8729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-01T22:45:43.362721+00:00,1a06cc53-5440-40c5-91b1-c670bc3bbd5d
1,35372567,3550308,municipal,2,1.0,1.291667,2026-08-21T14:53:54.662279+00:00,2026-09-01T22:43:25.497453+00:00,09680cbf-1ca4-4aa0-aad4-93f80d0fc395,44.37,51.06,57.72,64.11,70.03,75.35,80.0,1.0,basedosdados.br_inep_avaliacao_alfabetizacao.m...,639d5481-b968-4c35-b649-4a48b1d0da10,São Paulo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.28,739.4069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-01T22:45:43.362721+00:00,1a06cc53-5440-40c5-91b1-c670bc3bbd5d
2,35084816,3554508,municipal,2,1.0,1.294118,2026-08-21T14:53:54.662279+00:00,2026-09-01T22:43:25.497453+00:00,09680cbf-1ca4-4aa0-aad4-93f80d0fc395,53.53,58.64,63.56,68.22,72.54,76.47,80.0,3.0,basedosdados.br_inep_avaliacao_alfabetizacao.m...,639d5481-b968-4c35-b649-4a48b1d0da10,Tietê,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.28,739.4069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-01T22:45:43.362721+00:00,1a06cc53-5440-40c5-91b1-c670bc3bbd5d
3,33060533,3303500,municipal,2,0.0,1.300000,2026-08-21T14:53:54.662279+00:00,2026-09-01T22:43:25.497453+00:00,09680cbf-1ca4-4aa0-aad4-93f80d0fc395,40.13,47.44,54.87,62.08,68.80,74.81,80.0,0.0,basedosdados.br_inep_avaliacao_alfabetizacao.m...,639d5481-b968-4c35-b649-4a48b1d0da10,Nova Iguaçu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.13,737.3032,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-01T22:45:43.362721+00:00,1a06cc53-5440-40c5-91b1-c670bc3bbd5d
4,24012111,2413359,municipal,2,0.0,1.300000,2026-08-21T14:53:54.662279+00:00,2026-09-01T22:43:25.497453+00:00,09680cbf-1ca4-4aa0-aad4-93f80d0fc395,39.61,46.99,54.51,61.83,68.65,74.74,80.0,0.0,basedosdados.br_inep_avaliacao_alfabetizacao.m...,639d5481-b968-4c35-b649-4a48b1d0da10,Serra do Mel,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.39,723.8538,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-01T22:45:43.362721+00:00,1a06cc53-5440-40c5-91b1-c670bc3bbd5d


Tamanho da Base de Dados

In [ ]:
print(f"alunos_features (Linhas, Colunas): {alunos.shape}")
print(f"contexto_territorio (Linhas, Colunas): {contexto.shape}")
print(f"indicador município (Linhas, Colunas): {ind_mun.shape}")
print(f"indicador UF (Linhas, Colunas): {ind_uf.shape}")

Verificar os tipos dos Dados

In [ ]:
alunos.dtypes

Existe valores nulos?

In [ ]:
nulos = alunos.isnull().sum().sum()
print(f"Nulos em alunos_features (antes do join): {nulos}")

prof = profile(alunos)
prof.head(15)

> A Gold (Fase 2) deve entregar **fato + contexto já integrados** (`alunos_features` ou `alunos_analytic`). A Fase 3 apenas consome — sem join/normalização de `rede` ou chaves aqui.

Quais informações queremos trabalhar?

Selecionamos o target, chaves de junção e um conjunto enxuto de features de contexto (sem leakage).

In [ ]:
print("Colunas alunos_features:")
print(list(alunos.columns))

print("\nColunas que NÃO entram no modelo (leakage / IDs / metadados):")
print([c for c in LEAKAGE_COLS if c in alunos.columns])

In [ ]:
from src.config import GOLD_TABLE

# Tabela de modelagem — enriquecida na Gold (sem join manual)
df = build_eda_frame(tables)

if "sigla_uf" in df.columns:
    filled = df["sigla_uf"].notna().sum()
    print(f"Tabela `{GOLD_TABLE}`: {df.shape}")
    print(f"Linhas com sigla_uf: {filled} / {len(df)}")

df.head()

In [ ]:
df.tail()

Informações da Base (após o enriquecimento)

In [ ]:
df.info()

Valores únicos

In [ ]:
df.nunique()

Estatística Descritiva

In [ ]:
df.describe().round(2)

### Primeira análise — o target

**`alfabetizado`**
- `1` → aluno considerado alfabetizado  
- `0` → aluno não alfabetizado  

Essa é a variável que o modelo supervisionado deverá prever.

In [ ]:
print("Distribuição do target:")
display(target_balance(df))

plt.figure(figsize=(6, 4))
df[TARGET_COL].value_counts(dropna=False).plot(kind="bar", color=["#c44e52", "#4c72b0"])
plt.title("Distribuição — alfabetizado (amostra)")
plt.xlabel(TARGET_COL)
plt.ylabel("contagem")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_target_distribution.png", dpi=120)
plt.show()

Interessante observar se a amostra está equilibrada. Se uma classe dominar demais, na modelagem priorizamos métricas como **F1 / recall** e, se preciso, `class_weight`.

### Análise por rede e território

Pergunta: a taxa média de alfabetização na amostra muda entre redes e regiões?

In [ ]:
if "rede" in df.columns:
    taxa_rede = df.groupby("rede")[TARGET_COL].mean().sort_values(ascending=False)
    display(taxa_rede.to_frame("taxa_media_alfabetizado"))

    plt.figure(figsize=(8, 4))
    taxa_rede.plot(kind="bar", color=sns.color_palette("mako", len(taxa_rede)))
    plt.title("Taxa média de alfabetizado por rede (amostra)")
    plt.ylabel(f"média({TARGET_COL})")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "eda_rede_vs_target.png", dpi=120)
    plt.show()

In [ ]:
if "nome_regiao" in df.columns and df["nome_regiao"].notna().any():
    taxa_regiao = df.groupby("nome_regiao")[TARGET_COL].mean().sort_values(ascending=False)
    display(taxa_regiao.to_frame("taxa_media_alfabetizado"))

    plt.figure(figsize=(8, 4))
    taxa_regiao.plot(kind="bar", color=sns.color_palette("mako", len(taxa_regiao)))
    plt.title("Taxa média de alfabetizado por região (amostra)")
    plt.ylabel(f"média({TARGET_COL})")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "eda_nome_regiao_vs_target.png", dpi=120)
    plt.show()
else:
    print("nome_regiao sem valores preenchidos após o merge nesta amostra.")

Se houver diferença clara entre redes/regiões, isso reforça a hipótese de que **variáveis territoriais** devem entrar no modelo (com encoding categórico).

### Distribuições numéricas

Olhamos PIB per capita, IVS e lags — candidatas fortes a features.

In [ ]:
num_plot = [
    c
    for c in ["peso_aluno", "populacao", "pib_per_capita", "ivs", "lag1_taxa_alfabetizacao"]
    if c in df.columns and df[c].notna().any()
]

if num_plot:
    df[num_plot].hist(bins=15, figsize=(12, 6), color="#4c72b0", edgecolor="white")
    plt.suptitle("Distribuições numéricas (amostra enriquecida)")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "eda_numeric_distributions.png", dpi=120)
    plt.show()
else:
    print("Sem colunas numéricas preenchidas para histograma nesta amostra.")

### Pairplot / relações

Assim como no EDA de IDHM (renda × educação), aqui buscamos relações entre contexto socioeconômico e o target.

In [ ]:
pair_cols = [
    c
    for c in [TARGET_COL, "pib_per_capita", "ivs", "lag1_taxa_alfabetizacao", "populacao"]
    if c in df.columns and df[c].notna().any()
]

if len(pair_cols) >= 3:
    sns.pairplot(df[pair_cols].dropna(), hue=TARGET_COL if TARGET_COL in pair_cols else None)
    plt.show()
else:
    print("Amostra insuficiente de colunas preenchidas para pairplot.")

Interessante observar se `ivs` / `pib_per_capita` / `lag1_taxa_alfabetizacao` se separam visualmente entre alfabetizados e não alfabetizados. Isso guia a priorização de features.

### Correlações

Matriz de correlação entre variáveis numéricas relevantes e o target (amostra — provisória).

In [ ]:
cols_corr = [
    c
    for c in [
        TARGET_COL,
        "peso_aluno",
        "populacao",
        "pib_per_capita",
        "ivs",
        "ivs_capital_humano",
        "lag1_taxa_alfabetizacao",
    ]
    if c in df.columns and df[c].notna().any()
]

if len(cols_corr) >= 2:
    corr = df[cols_corr].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="rocket", vmin=-1, vmax=1)
    plt.title("Matriz de correlação — target e contexto")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "eda_correlation_heatmap.png", dpi=120)
    plt.show()

    print("Correlação com o target (amostra):")
    display(correlation_with_target(df))
else:
    print("Poucas colunas numéricas preenchidas para correlação nesta amostra.")

### Indicadores agregados (município e UF)

Essas bases respondem perguntas de **negócio** (metas, risco municipal), não o grain do classificador de aluno.

In [ ]:
print("Indicador município — preview")
display(ind_mun.head(3))
display(ind_mun.describe().round(2).T.head(12))

if "taxa_crianca_alfabetizada" in ind_mun.columns:
    plt.figure(figsize=(7, 4))
    ind_mun["taxa_crianca_alfabetizada"].dropna().hist(bins=15, color="#55a868", edgecolor="white")
    plt.title("Distribuição — taxa_crianca_alfabetizada (município, amostra)")
    plt.tight_layout()
    plt.show()

In [ ]:
print("Indicador UF — preview")
display(ind_uf.head(5))

if {"sigla_uf", "taxa_crianca_alfabetizada"}.issubset(ind_uf.columns):
    tmp = ind_uf.dropna(subset=["taxa_crianca_alfabetizada"]).sort_values(
        "taxa_crianca_alfabetizada", ascending=False
    )
    plt.figure(figsize=(12, 5))
    pal = sns.color_palette("mako", len(tmp))
    plt.bar(tmp["sigla_uf"].astype(str), tmp["taxa_crianca_alfabetizada"], color=pal, width=0.9)
    plt.title("UF × taxa_crianca_alfabetizada (amostra)")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "eda_uf_taxa.png", dpi=120)
    plt.show()

Esse tipo de visão por UF ajuda a responder: *quais estados/municípios apresentam maior risco educacional?* — pergunta estratégica do Tech Challenge.

### Features candidatas vs leakage

Resumo do que deve (e não deve) ir para a pipeline Scikit-learn.

In [ ]:
feats = feature_columns(df)
num, cat = split_feature_types(df, feats)

print(f"Features candidatas: {len(feats)}")
print(f"Numéricas ({len(num)}):", num)
print(f"Categóricas ({len(cat)}):", cat)
print("\nExcluir do modelo:", [c for c in LEAKAGE_COLS if c in alunos.columns or c in df.columns])

### Hipóteses analíticas

| # | Hipótese | Variáveis | Implicação para modelagem |
|---|----------|-----------|---------------------------|
| H1 | Contexto socioeconômico influencia a chance de alfabetização | `ivs*`, `pib_per_capita`, `populacao` | Incluir + imputar missing |
| H2 | Histórico municipal (lag) é preditivo com menor risco de leakage | `lag1_*` | Preferir lags a taxas do mesmo evento |
| H3 | Região e rede escolar alteram a taxa de alfabetização | `nome_regiao`, `sigla_uf`, `rede` | One-hot / encoding |
| H4 | Metas sozinhas não explicam o aluno | `meta_alfabetizacao_*` | Usar com cautela |

**Decisões de modelagem (a partir desta EDA):**
1. Grain = **aluno**; target = **`alfabetizado`**
2. Enriquecer fato aluno com `contexto_territorio` (`id_municipio`, `rede`)
3. Remover `id_aluno` e metadados `_silver_*` / `_gold_*`; usar `nivel_alfabetizacao` como feature se a Gold expuser
4. Pipeline: imputação + scaling (num) e imputação + one-hot (cat)
5. Revalidar em amostra maior / Gold S3 antes do treino final

In [ ]:
images = sorted(IMAGES_DIR.glob("eda_*.png"))
report = write_eda_report(tables, images, enriched=df)
print("Relatório escrito em:", report)